# Legal corpus build on Kaggle: laws + regulations -> ready ChromaDB

One run does everything and leaves the finished database in this version's **Output** tab:
1. **Fetch** (with a progress line per step): the Knesset registry tables, then the laws and the regulations from the
   Hebrew Wikisource Open Law Book (official Wikimedia dump) -- `scripts/legal_data/fetch_*.py`.
2. **Chunk + vectorize** on the GPU with bge-m3 -- `scripts/legal_data/vectorize.py`.
3. **Package**: `legal_corpus_vectordb.zip` (the database: `laws/`, `procedural_rules/`, `_build_info.json`) and
   `legal_corpus_jsonl.zip` (the normalized records, reports and run manifests).

**Before running** (panel on the right):
1. *Settings -> Accelerator*: **GPU T4 x2** (one GPU is used by default). *Settings -> Internet*: **On**.
2. *Add-ons -> Secrets*: tick **GITHUB_TOKEN** (can read `zananiri/AI-IZ`) and add **LEGAL_DATA_CONTACT_EMAIL**
   (your email -- the fetchers put it in their User-Agent and refuse to run without it).
3. In the next cell, set `CHROMADB_VERSION` to the chromadb version on the computer you'll install on
   (`python -c "import chromadb; print(chromadb.__version__)"`) so the files open there.
4. To skip the download, attach an earlier run's output: *Add Input -> Your Work -> Notebooks* -> that
   notebook (its `AI-IZ/legal_txt/`), or a dataset of `legal_corpus_jsonl.zip`. `REUSE_FETCH` picks it up.
5. *Save Version -> Save & Run All (Commit)*. Expect roughly 1-3 hours; each script prints progress with an ETA.
   The run stops at the first failed step (no GPU, a failed fetch or vectorize, an empty database) instead of
   packaging whatever is there -- the error is at the end of that cell's output.

**Install on your computer** when it finishes: download `legal_corpus_vectordb.zip` from *Output*, then run
`python scripts/legal_data/install_corpus.py <path to the zip>` -- it unpacks into `data/legal_corpus_vectordb/`
and checks every collection's chunk count.

In [ ]:
BRANCH = "main"
CATEGORIES = "laws,procedural_rules"   # laws = Basic Laws, laws, ordinances; procedural_rules = regulations etc.
CHROMADB_VERSION = "1.5.9"             # the chromadb version on the computer you'll install on
DEVICE = "cuda"                        # one T4; "cuda:0,cuda:1" uses both (sentence-transformers process pool)
DUMP_DATE = "latest"                   # newest completed Wikisource dump, or pin e.g. "20260901"
REUSE_FETCH = True                     # use the records of an attached earlier run (Add Input -> its Output)
                                       # instead of downloading again; downloads if none is attached

## 1. Code, packages and settings

In [ ]:
import os, pathlib, shutil, json, glob, subprocess
from kaggle_secrets import UserSecretsClient

# Before the long fetch: the embedding step needs the GPU(s) DEVICE names.
if DEVICE.startswith("cuda"):
    import torch
    wanted = len(DEVICE.split(","))
    if torch.cuda.device_count() < wanted:
        raise RuntimeError(f"DEVICE={DEVICE!r} needs {wanted} GPU(s), found {torch.cuda.device_count()}: "
                           "Settings -> Accelerator -> GPU T4 x2, then run again")

def sh(command):
    """Runs a shell command with its output in this cell; raises (stopping the run) if it fails.
    `!command` would carry on after a failure."""
    proc = subprocess.Popen(["bash", "-o", "pipefail", "-c", command], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"exit code {proc.returncode}: {command}")

secrets = UserSecretsClient()
token = secrets.get_secret("GITHUB_TOKEN")
os.environ["DOCSLIDES_LEGAL_DATA_CONTACT_EMAIL"] = secrets.get_secret("LEGAL_DATA_CONTACT_EMAIL")
REPO = "/kaggle/working/AI-IZ"
!git clone -q --depth 1 -b {BRANCH} https://{token}@github.com/zananiri/AI-IZ.git {REPO}
if not pathlib.Path(REPO, ".git").exists():
    raise RuntimeError("git clone failed: check the GITHUB_TOKEN secret and BRANCH")
!git -C {REPO} remote set-url origin https://github.com/zananiri/AI-IZ.git
%cd {REPO}
sh("git log --oneline -1")
sh('pip install -q -e ".[legal,legal-data,dev]"')
sh(f'pip install -q "chromadb=={CHROMADB_VERSION}"')
sh("python -c \"import chromadb, sentence_transformers; print('chromadb', chromadb.__version__, '| sentence-transformers', sentence_transformers.__version__)\"")

if DUMP_DATE != "latest":  # pin the Wikisource dump
    import yaml
    cfg = yaml.safe_load(open("config/config.yaml", encoding="utf-8"))
    cfg["legal_data"]["sources"]["wikisource_dump_date"] = DUMP_DATE
    yaml.safe_dump(cfg, open("config/kaggle_corpus.yaml", "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
    os.environ["DOCSLIDES_CONFIG"] = "config/kaggle_corpus.yaml"
# The database is built in scratch space; only the zips go to /kaggle/working (the saved Output).
VECTORDB = "/tmp/legal_corpus_vectordb"
pathlib.Path(VECTORDB).mkdir(parents=True, exist_ok=True)
DEVICE_ARG = f"--devices {DEVICE}" if "," in DEVICE else f"--device {DEVICE}"

## 2. Unit tests (about a minute; informational -- a failure here doesn't stop the run)

In [ ]:
!python -m pytest tests/unit -q -p no:cacheprovider 2>&1 | tail -5

## 3. Knesset registry: the four tables the laws and regulations join to (or an earlier run's records, reused)

In [ ]:
REUSED = None
if REUSE_FETCH:
    found = sorted(pathlib.Path("/kaggle/input").rglob("laws/laws*.jsonl"))
    if found:
        REUSED = found[0].parent.parent
        for sub in ("laws", "procedural_rules", "metadata", "_manifests"):
            if (REUSED / sub).is_dir():  # not laws/raw/: the Wikisource dump isn't needed again
                shutil.copytree(REUSED / sub, pathlib.Path("legal_txt") / sub, dirs_exist_ok=True,
                                ignore=shutil.ignore_patterns("raw"))
        print("reusing the records in", REUSED, "-- the download steps are skipped")
    else:
        print("REUSE_FETCH: no laws/laws*.jsonl under /kaggle/input, downloading")
if not REUSED:
    sh("python scripts/legal_data/fetch_metadata.py --tables KNS_IsraelLaw,KNS_IsraelLawName,KNS_SecondaryLaw,KNS_SecLawAuthorizingLaw")

## 4. Laws and regulations from the Open Law Book

The first script downloads the dump (sha1-verified); the second reuses it. Registry PDFs are skipped (`--pdfs none`):
the text comes from the Open Law Book, and the regulation PDFs are mostly committee background material.

In [ ]:
if not REUSED:
    if "laws" in CATEGORIES:
        sh("python scripts/legal_data/fetch_laws.py --pdfs none")
    if "procedural_rules" in CATEGORIES:
        sh("python scripts/legal_data/fetch_procedural_rules.py --pdfs none")

## 5. What was fetched

In [ ]:
for path in sorted(glob.glob("legal_txt/_manifests/*_fetch_*.json")):
    run = json.load(open(path, encoding="utf-8"))
    print(f"{run['script']}: {run['status']} | counts {run['counts']}")
    for warning in run["warnings"][:5]:
        print("   warning:", warning)
if os.path.exists("legal_txt/procedural_rules/coverage_report.json"):
    report = json.load(open("legal_txt/procedural_rules/coverage_report.json", encoding="utf-8"))
    for label, found in report["required"].items():
        print(f"required {label}: {len(found['found'])} found", found["found"][:3])
!ls -la legal_txt/laws legal_txt/procedural_rules 2>/dev/null | grep jsonl

## 6. Chunk and vectorize (resumable: re-running this cell continues where it stopped)

In [ ]:
sh(f"python scripts/legal_data/vectorize.py --input legal_txt --vectordb {VECTORDB} --categories {CATEGORIES} --dry-run")
sh(f"python scripts/legal_data/vectorize.py --input legal_txt --vectordb {VECTORDB} --categories {CATEGORIES} {DEVICE_ARG}")

## 7. Test queries with metadata filters

In [ ]:
PROBES = [("laws", "מה דינו של חוזה שנכרת בטעות?", '{"status": "in_force"}'),
          ("laws", "מי רשאי להגיש רשימת מועמדים לכנסת?", ""),
          ("procedural_rules", "מה המועד להגשת כתב הגנה?", ""),
          ("procedural_rules", "מה גובה האגרה בערעור?", '{"authority_level": "regulation"}')]
for category, question, where in PROBES:
    if category in CATEGORIES:
        where_arg = f"--where '{where}'" if where else ""
        sh(f'python scripts/legal_data/vectorize.py --vectordb {VECTORDB} --probe "{question}" --category {category} {where_arg} --top-k 3')

## 8. Output: the database and the records

In [ ]:
sh(f"python scripts/legal_data/vectorize.py --vectordb {VECTORDB} --categories {CATEGORIES} --export-lexical")
info = json.load(open(f"{VECTORDB}/_build_info.json", encoding="utf-8"))
empty = [c for c in CATEGORIES.split(",") if not info["categories"].get(c, {}).get("chunks")]
if empty:
    raise RuntimeError(f"no chunks for {empty}: not packaging an empty database")
shutil.make_archive("/kaggle/working/legal_corpus_vectordb", "zip", VECTORDB)

records = pathlib.Path("/tmp/legal_corpus_jsonl")
for pattern in ("laws/*.jsonl", "laws/*.json", "procedural_rules/*.jsonl", "procedural_rules/*.json",
                "metadata/knesset/*", "_manifests/*.json", "_manifests/*.log"):
    for path in pathlib.Path("legal_txt").glob(pattern):
        target = records / path.relative_to("legal_txt")
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, target)
shutil.make_archive("/kaggle/working/legal_corpus_jsonl", "zip", records)

print(json.dumps(info, ensure_ascii=False, indent=1))
sh("ls -lh /kaggle/working/*.zip")
print("On your computer: python scripts/legal_data/install_corpus.py <path to legal_corpus_vectordb.zip>")